In [0]:
%pip install xgboost

In [0]:
"""
04_Train_Model_v8.py — Notebook de entrenamiento unificado
===========================================================
Consolida las 4 celdas anteriores en un único entrenamiento:
  - Lee Gold app_inmuebles/ (tiene comunas+sectores+market_token calculados por Spark)
  - market_token como feature categórica (no recalcula, viene del Gold)
  - portal ops weighting integrado (no re-entrena, ajusta pesos antes del split)
  - amenity regex corregido (\\b → r"\b")
  - sample_weight calculado sobre df_clean con índices alineados
  - bundle_v8 con todos los stats + trazabilidad de fuente
"""

import pandas as pd
import numpy as np
import pickle
import boto3
import json
import re
import time
import unicodedata
import sys
import os
from io import BytesIO
from datetime import datetime, timezone

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
from xgboost import XGBRegressor

# ── Control de promoción ──────────────────────────────────────────
AUTO_PROMOTE_IF_BETTER        = True
FORCE_PROMOTE_TO_CHAMPION     = True
USE_HISTORY_FEATURES          = True
HISTORY_MIN_REPEATED_SHARE    = 1.0

# =============================================================
# 1. CONFIG Y CARGA
# =============================================================

# ═══════════════════════════════════════════════════════════
# BUILD _candidates FIRST (needed for credentials)
# ═══════════════════════════════════════════════════════════
_candidates = []
try:
    _nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _repo_dir = "/Workspace" + str(_nb_path).rsplit("/", 2)[0]
    _candidates.append(_repo_dir)
except Exception:
    pass
_vsc_file = globals().get("__vsc_ipynb_file__", "")
if _vsc_file:
    _nb_dir = os.path.dirname(os.path.abspath(_vsc_file))
    _candidates.append(_nb_dir)
    _parent = os.path.dirname(_nb_dir)
    if _parent and _parent != _nb_dir:
        _candidates.append(_parent)
_candidates.append(os.getcwd())
for _candidate in _candidates:
    if _candidate and _candidate not in sys.path:
        sys.path.insert(0, _candidate)

try:
    config = {
        "aws_access_key": dbutils.secrets.get(scope="aws", key="access_key"),
        "aws_secret_key": dbutils.secrets.get(scope="aws", key="secret_key"),
        "bucket_name": "bronce-scrap-date",
    }
except Exception:
    try:
        _aws_file = next(
            (os.path.join(d, "aws_secrets.json") for d in _candidates
             if d and os.path.isfile(os.path.join(d, "aws_secrets.json"))),
            "aws_secrets.json"
        )
        with open(_aws_file, "r") as f:
            config = json.load(f)
    except Exception:
        with open("aws_secrets.json", "r") as f:
            config = json.load(f)

BUCKET = config["bucket_name"]

# SIN fs.s3a.impl — no compatible con Databricks Serverless
S3_OPTS = {
    "fs.s3a.access.key": config["aws_access_key"],
    "fs.s3a.secret.key": config["aws_secret_key"],
    "fs.s3a.endpoint":   "s3.amazonaws.com",
}

# Prioridad de fuente de training:
# 1. Gold app_inmuebles (tiene comunas+sectores+market_token del pipeline Spark)
# 2. Silver master_deduped (fallback — requiere recalcular geo en Python)
GOLD_PATH    = f"s3a://{BUCKET}/gold/app_inmuebles/"
SILVER_PATH  = f"s3a://{BUCKET}/silver/master_deduped/"
HISTORY_PATH = f"s3a://{BUCKET}/silver/master_inmuebles/"
PORTAL_OPS_PATH = f"s3a://{BUCKET}/gold/portal_operacion/"

def summarize_spark_exception(exc):
    msg = re.sub(r"\s+", " ", str(exc)).strip()
    for tok in ["JVM stacktrace:", "Caused by:", "Trace ID:"]:
        if tok in msg:
            msg = msg.split(tok, 1)[0].strip()
    return msg[:280] + ("..." if len(msg) > 280 else "")

def load_spark_path(path, formats):
    last_error = None
    for fmt in formats:
        try:
            reader = spark.read.format(fmt)
            for k, v in S3_OPTS.items():
                reader = reader.option(k, v)
            return reader.load(path), fmt
        except Exception as exc:
            last_error = exc
    raise last_error

# ── Cargar datos de entrenamiento ────────────────────────────────
training_source = "gold"
try:
    df_spark, fmt = load_spark_path(GOLD_PATH, ["parquet", "delta"])
    print(f"✅ Leyendo Gold ({fmt.upper()}): {GOLD_PATH}")
except Exception as e:
    print(f"⚠️ Gold no disponible ({summarize_spark_exception(e)}), usando Silver Deduped")
    df_spark, fmt = load_spark_path(SILVER_PATH, ["delta"])
    training_source = "silver"
    print(f"✅ Leyendo Silver Deduped (delta): {SILVER_PATH}")

available = set(df_spark.columns)
GEO_COLS = ["comuna_mercado", "sector_mercado", "market_token", "zona_mercado"]
geo_present = [c for c in GEO_COLS if c in available]
geo_missing  = [c for c in GEO_COLS if c not in available]
print(f"   Columnas geo del Gold presentes: {geo_present}")
if geo_missing:
    print(f"   ⚠️ Ausentes (se recalcularán): {geo_missing}")

# ── Cargar historial ─────────────────────────────────────────────
df_history_spark = None
history_read_enabled = USE_HISTORY_FEATURES
if history_read_enabled:
    try:
        df_history_spark, hfmt = load_spark_path(HISTORY_PATH, ["delta", "parquet"])
        print(f"🕒 Historial ({hfmt.upper()}): {HISTORY_PATH}")
    except Exception as e:
        history_read_enabled = False
        print(f"⚠️ Historial deshabilitado: {summarize_spark_exception(e)}")

# ── Cargar portal ops ────────────────────────────────────────────
portal_ops = pd.DataFrame()
try:
    df_portal_ops_spark, pfmt = load_spark_path(PORTAL_OPS_PATH, ["delta", "parquet"])
    portal_cols = [c for c in [
        "portal", "checkpoint_activo", "checkpoint_age_hours",
        "portal_ofertas_activas", "pct_multiportal_portal",
        "dispersion_promedio_portal_pct", "portal_health_score",
    ] if c in set(df_portal_ops_spark.columns)]
    portal_ops = df_portal_ops_spark.select(*portal_cols).toPandas()
    portal_ops["portal"] = portal_ops["portal"].fillna("").astype(str).str.strip().str.lower()
    for col in portal_cols[1:]:
        portal_ops[col] = pd.to_numeric(portal_ops[col], errors="coerce")
    print(f"🛰️ Portal ops ({pfmt.upper()}): {len(portal_ops)} portales")
except Exception as e:
    print(f"ℹ️ Portal ops no disponible: {summarize_spark_exception(e)}")

# =============================================================
# 2. PREPARAR FEATURES BASE
# =============================================================
base_numeric_candidates = [
    "area_m2", "habitaciones", "banos", "garajes",
    "num_portales", "dispersion_pct_grupo", "precio_desviacion_grupo_pct",
    "data_completeness",
]
cols_numeric_base_raw = [c for c in base_numeric_candidates if c in available]

# Columnas geo — del Gold si existen, sino se recalculan más abajo
geo_cat_candidates = ["tipo_inmueble", "estado_inmueble", "fuente", "city_token",
                       "comuna_mercado", "sector_mercado", "market_token"]
cols_text_candidates = ["ubicacion_norm", "zona_mercado", "ubicacion_raw", "titulo"]
cols_text = [c for c in cols_text_candidates if c in available]
aux_cols = [c for c in ["property_group_id", "id_original", "fecha_extraccion",
                          "url", "batch_id", "source_file"] if c in available]

all_cols = list(dict.fromkeys(
    ["precio_num"] + cols_numeric_base_raw +
    [c for c in geo_cat_candidates if c in available] +
    cols_text + aux_cols
))

df = df_spark.select(*all_cols).toPandas()
print(f"   Total registros: {len(df):,}")

for col in cols_numeric_base_raw + ["precio_num"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
if "fecha_extraccion" in df.columns:
    df["fecha_extraccion"] = pd.to_datetime(df["fecha_extraccion"], errors="coerce")
for col in [c for c in geo_cat_candidates if c in df.columns] + cols_text + ["id_original", "url", "batch_id"]:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str)

# ── Normalización de texto ───────────────────────────────────────
def normalize_text(value):
    text = "" if pd.isna(value) else str(value)
    text = text.lower().strip()
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"\|.*", "", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def normalize_url(value):
    text = "" if pd.isna(value) else str(value).strip().lower()
    text = re.sub(r"\?.*$", "", text)
    return re.sub(r"/$", "", text)

def build_listing_key(frame):
    fuente_part = frame.get("fuente", pd.Series("desconocido", index=frame.index)).fillna("desconocido").astype(str)
    id_part     = frame["id_original"].fillna("").astype(str).str.strip() if "id_original" in frame.columns else pd.Series("", index=frame.index)
    url_part    = frame["url"].fillna("").astype(str).apply(normalize_url) if "url" in frame.columns else pd.Series("", index=frame.index)
    group_part  = frame["property_group_id"].fillna("").astype(str).str.strip() if "property_group_id" in frame.columns else pd.Series("", index=frame.index)
    return pd.Series(np.where(
        id_part.ne(""),    fuente_part + "__id__"  + id_part,
        np.where(url_part.ne(""),  fuente_part + "__url__" + url_part,
        np.where(group_part.ne(""), "group__" + group_part, ""))
    ), index=frame.index)

# ── Defaults para columnas faltantes ────────────────────────────
for col, default in [("city_token","otra_ciudad"),("tipo_inmueble","otro"),
                      ("fuente","desconocido"),("estado_inmueble","desconocido")]:
    if col not in df.columns:
        df[col] = default

tipos_validos = df["tipo_inmueble"].value_counts()
df["tipo_inmueble"] = df["tipo_inmueble"].where(
    df["tipo_inmueble"].isin(tipos_validos[tipos_validos >= 40].index), other="otro"
)

# ── Texto unificado ──────────────────────────────────────────────
text_parts = [df[c].fillna("") for c in cols_text if c in df.columns]
df["texto_completo"] = text_parts[0] if text_parts else pd.Series("", index=df.index)
for part in text_parts[1:]:
    df["texto_completo"] = df["texto_completo"] + " " + part

# ── Geo: usar Gold si está presente, sino recalcular ────────────
CITY_ALIAS_TO_TOKEN = {
    "bogota":        ["bogota", "bogota dc", "bogota d c", "bogota distrito capital"],
    "medellin":      ["medellin", "medallo"],
    "cali":          ["cali", "santiago de cali"],
    "barranquilla":  ["barranquilla", "bquilla"],
    "cartagena":     ["cartagena", "cartagena de indias"],
    "bucaramanga":   ["bucaramanga"],
    "pereira":       ["pereira"],
    "manizales":     ["manizales"],
    "cucuta":        ["cucuta", "san jose de cucuta"],
    "santa marta":   ["santa marta"],
}
CITY_ALIAS_TOKENS = {
    tok for aliases in CITY_ALIAS_TO_TOKEN.values()
    for alias in aliases for tok in alias.split()
}
CITY_ALIAS_TOKENS.update(CITY_ALIAS_TO_TOKEN.keys())

CIUDAD_A_MARKET = {
    "bogota": "bogota_metropolitana",    "soacha": "bogota_metropolitana",
    "chia": "bogota_metropolitana",      "cajica": "bogota_metropolitana",
    "zipaquira": "bogota_metropolitana", "cota": "bogota_metropolitana",
    "mosquera": "bogota_metropolitana",  "la calera": "bogota_metropolitana",
    "medellin": "valle_aburra",          "envigado": "valle_aburra",
    "sabaneta": "valle_aburra",          "itagui": "valle_aburra",
    "bello": "valle_aburra",
    "rionegro": "oriente_antioqueno",
    "cali": "cali_metropolitana",
    "barranquilla": "barranquilla_metropolitana",
    "cartagena": "cartagena_metropolitana",
    "bucaramanga": "bucaramanga_metropolitana",
    "floridablanca": "bucaramanga_metropolitana",
    "pereira": "eje_cafetero",           "manizales": "eje_cafetero",
    "armenia": "eje_cafetero",
    "santa marta": "santa_marta_metropolitana",
    "cucuta": "cucuta_metropolitana",
    "ibague": "ibague_metropolitana",
}

def canonicalize_city(raw):
    val = normalize_text(raw)
    if not val or val == "otra_ciudad":
        return "otra_ciudad"
    for token, aliases in CITY_ALIAS_TO_TOKEN.items():
        if val == token or any(a in val for a in aliases):
            return token
    return val

ubicacion_col = next((c for c in ["ubicacion_norm", "zona_mercado", "ubicacion_raw"] if c in df.columns), None)
df["ubicacion_limpia"] = df[ubicacion_col].apply(normalize_text) if ubicacion_col else ""

# city_token: normalizar aunque venga del Gold (puede tener variantes)
df["city_token"] = df["city_token"].apply(canonicalize_city)
conteos = df["city_token"].value_counts()
df["city_token"] = df["city_token"].where(
    df["city_token"].isin(conteos[conteos >= 80].index), other="otra_ciudad"
)
print(f"   city_token: {df['city_token'].nunique()} categorías")

# market_token: del Gold si existe, sino derivar
if "market_token" in df.columns and df["market_token"].ne("").any():
    mkt_counts = df["market_token"].value_counts()
    df["market_token"] = df["market_token"].where(
        df["market_token"].isin(mkt_counts[mkt_counts >= 50].index), other="mercado_otro"
    )
    print(f"   market_token del Gold: {df['market_token'].nunique()} categorías")
else:
    df["market_token"] = df["city_token"].map(CIUDAD_A_MARKET).fillna("mercado_otro")
    print(f"   market_token derivado de city_token: {df['market_token'].nunique()} categorías")

# comunas y sectores: del Gold si existen
from src.geo.sector_mapping import assign_comuna as assign_comuna_py, extract_sector_mercado as extract_sector_py

if "comuna_mercado" not in df.columns or df["comuna_mercado"].eq("").all():
    df["comuna_mercado"] = [assign_comuna_py(c, u) for c, u in zip(df["city_token"], df["ubicacion_limpia"])]
    # colapsar
    com_counts = df["comuna_mercado"].value_counts()
    df["comuna_mercado"] = df["comuna_mercado"].where(
        df["comuna_mercado"].isin(com_counts[com_counts >= 25].index), other="comuna_otra"
    )
    print(f"   comuna_mercado recalculada: {df['comuna_mercado'].nunique()} categorías")
else:
    com_counts = df["comuna_mercado"].value_counts()
    df["comuna_mercado"] = df["comuna_mercado"].where(
        df["comuna_mercado"].isin(com_counts[com_counts >= 15].index), other="comuna_otra"
    )
    print(f"   comuna_mercado del Gold: {df['comuna_mercado'].nunique()} categorías")

if "sector_mercado" not in df.columns or df["sector_mercado"].eq("").all():
    df["sector_mercado_raw"] = [extract_sector_py(c, com, u)
                                 for c, com, u in zip(df["city_token"], df["comuna_mercado"], df["ubicacion_limpia"])]
    sc = df.groupby(["city_token","sector_mercado_raw"]).size().reset_index(name="sn")
    df = df.merge(sc, on=["city_token","sector_mercado_raw"], how="left")
    df["sector_mercado"] = np.where(
        df["sn"].fillna(0) >= 12, df["sector_mercado_raw"],
        np.where(df["comuna_mercado"].ne("comuna_otra"), df["comuna_mercado"], "sector_otra")
    )
    df = df.drop(columns=["sector_mercado_raw","sn"], errors="ignore")
    print(f"   sector_mercado recalculado: {df['sector_mercado'].nunique()} categorías")
else:
    sec_counts = df["sector_mercado"].value_counts()
    df["sector_mercado"] = df["sector_mercado"].where(
        df["sector_mercado"].isin(sec_counts[sec_counts >= 12].index), other="sector_otra"
    )
    print(f"   sector_mercado del Gold: {df['sector_mercado'].nunique()} categorías")

df["listing_history_key"] = build_listing_key(df)

# =============================================================
# 3. FEATURES DE CONSENSO Y AMENITIES
# =============================================================
num_portales_n = df["num_portales"].fillna(1).clip(1, 4) if "num_portales" in df.columns else pd.Series(1.0, index=df.index)
dispersion     = df["dispersion_pct_grupo"].fillna(50).clip(0, 100) if "dispersion_pct_grupo" in df.columns else pd.Series(50.0, index=df.index)
desv           = df["precio_desviacion_grupo_pct"].fillna(50).clip(0, 100) if "precio_desviacion_grupo_pct" in df.columns else pd.Series(50.0, index=df.index)
completeness   = df["data_completeness"].fillna(0.5) if "data_completeness" in df.columns else pd.Series(0.5, index=df.index)
if completeness.max(skipna=True) > 1.5:
    completeness = completeness / 100.0
completeness = completeness.clip(0.0, 1.0)

df["score_consenso_cross_portal"] = (
    np.clip((num_portales_n - 1.0) / 3.0, 0, 1) * 35.0
    + (1.0 - dispersion / 100.0) * 30.0
    + (1.0 - desv / 100.0) * 20.0
    + completeness * 15.0
).clip(0.0, 100.0)

# CORREGIDO: r"\b..." en lugar de "\\b..." (el doble backslash nunca matcheaba)
AMENITY_PATTERNS = {
    "amenity_balcon":     ["balcon", "balcony"],
    "amenity_terraza":    ["terraza", "roof garden", "patio"],
    "amenity_ascensor":   ["ascensor", "elevador"],
    "amenity_deposito":   ["deposito", "bodega", "locker", "cuarto util"],
    "amenity_estudio":    ["estudio", "biblioteca"],
    "amenity_remodelado": ["remodelado", "renovado", "reformado", "para estrenar"],
    "amenity_gimnasio":   ["gimnasio", "gym"],
    "amenity_piscina":    ["piscina", "pool", "jacuzzi"],
    "amenity_conjunto":   ["conjunto", "unidad cerrada", "urbanizacion cerrada"],
    "amenity_vigilancia": ["vigilancia", "porteria", "seguridad"],
    "amenity_vista":      ["vista panoramica", "vista al mar", "vista verde"],
    "amenity_penthouse":  ["penthouse"],
    "amenity_duplex":     ["duplex", "triplex"],
    "amenity_amoblado":   ["amoblado", "full amoblado"],
    "amenity_lujo":       ["lujo", "exclusivo", "premium", "alta valorizacion"],
}
text_signal = (df["texto_completo"].fillna("") + " " + df["ubicacion_limpia"].fillna("")).apply(normalize_text)
amenity_feature_cols = []
for fname, keywords in AMENITY_PATTERNS.items():
    # r"\b" correcto — matchea word boundary real
    pattern = r"\b(?:" + "|".join(re.escape(kw) for kw in keywords) + r")\b"
    df[fname] = text_signal.str.contains(pattern, regex=True, na=False).astype(float)
    amenity_feature_cols.append(fname)
df["amenities_count"]  = df[amenity_feature_cols].sum(axis=1)
df["amenity_lujo_score"] = df[["amenity_balcon","amenity_terraza","amenity_ascensor",
                                 "amenity_gimnasio","amenity_piscina","amenity_vista","amenity_lujo"]].sum(axis=1)
print(f"   amenities: cobertura media = {df['amenities_count'].gt(0).mean()*100:.1f}%")

# =============================================================
# 4. HISTORIAL
# =============================================================
history_feature_cols = [
    "dias_en_mercado","republicaciones_count","num_repricings",
    "descuento_desde_inicial_hist_pct","dias_desde_ultimo_repricing",
    "n_observaciones_hist","n_batches_hist","n_urls_hist",
]
history_repeated_share = 0.0
history_signal_enabled = False

if USE_HISTORY_FEATURES and df_history_spark is not None:
    h_available = set(df_history_spark.columns)
    history_cols = [c for c in ["fuente","id_original","url","property_group_id",
                                  "fecha_extraccion","precio_num","batch_id","source_file"]
                    if c in h_available]
    try:
        ht0 = time.time()
        hdf = df_history_spark.select(*history_cols).toPandas()
        for col in ["fuente","id_original","url","property_group_id","batch_id","source_file"]:
            if col in hdf.columns:
                hdf[col] = hdf[col].fillna("").astype(str)
        if "fecha_extraccion" in hdf.columns:
            hdf["fecha_extraccion"] = pd.to_datetime(hdf["fecha_extraccion"], errors="coerce")
        if "precio_num" in hdf.columns:
            hdf["precio_num"] = pd.to_numeric(hdf["precio_num"], errors="coerce")
        hdf["listing_history_key"] = build_listing_key(hdf)
        hdf = hdf[hdf["listing_history_key"].ne("") & hdf["fecha_extraccion"].notna()].copy()
        hdf = hdf.sort_values(["listing_history_key","fecha_extraccion"], kind="stable")

        hcounts = hdf["listing_history_key"].value_counts()
        history_repeated_share = float((hcounts > 1).mean() * 100) if len(hcounts) else 0.0
        history_signal_enabled = history_repeated_share >= HISTORY_MIN_REPEATED_SHARE
        print(f"   historial: {len(hdf):,} registros, {history_repeated_share:.1f}% con >1 obs")

        if history_signal_enabled:
            hbase = hdf.groupby("listing_history_key").agg(
                first_date=("fecha_extraccion","min"),
                last_date=("fecha_extraccion","max"),
                n_observaciones_hist=("fecha_extraccion","size"),
            ).reset_index()

            hdf["gap_days"] = hdf.groupby("listing_history_key")["fecha_extraccion"].diff().dt.days
            gap_stats = hdf.groupby("listing_history_key", as_index=False)["gap_days"].apply(
                lambda x: (x > 45).sum()
            ).rename(columns={"gap_days":"gap_count"})

            url_stats = (hdf.assign(url_ne=hdf["url"].where(hdf["url"].ne(""), pd.NA))
                          .groupby("listing_history_key", as_index=False)["url_ne"].nunique()
                          .rename(columns={"url_ne":"n_urls_hist"}))
            batch_stats = (hdf.assign(batch_ne=hdf["batch_id"].where(hdf["batch_id"].ne(""), pd.NA))
                            .groupby("listing_history_key", as_index=False)["batch_ne"].nunique()
                            .rename(columns={"batch_ne":"n_batches_hist"}))

            hbase = hbase.merge(gap_stats, on="listing_history_key", how="left")
            hbase = hbase.merge(url_stats, on="listing_history_key", how="left")
            hbase = hbase.merge(batch_stats, on="listing_history_key", how="left")
            hbase["republicaciones_count"] = hbase["gap_count"].fillna(0).astype(int)

            hprice = hdf[["listing_history_key","fecha_extraccion","precio_num"]].dropna(subset=["precio_num"]).copy()
            hfirst = hprice.groupby("listing_history_key", as_index=False).first().rename(columns={"precio_num":"precio_inicial"})
            hlast  = hprice.groupby("listing_history_key", as_index=False).last().rename(columns={"precio_num":"precio_final","fecha_extraccion":"fecha_final"})
            hprice_stats = hfirst.merge(hlast, on="listing_history_key")
            hprice_stats["num_repricings"] = hprice.groupby("listing_history_key").apply(
                lambda x: (x["precio_num"].diff().ne(0).sum()) if len(x) > 1 else 0
            ).reset_index(name="num_repricings")["num_repricings"]
            hprice_stats["descuento_desde_inicial_hist_pct"] = (
                (hprice_stats["precio_inicial"] - hprice_stats["precio_final"]) / hprice_stats["precio_inicial"] * 100
            ).clip(lower=-300, upper=100)

            hbase = hbase.merge(hprice_stats[["listing_history_key","num_repricings",
                                               "descuento_desde_inicial_hist_pct","fecha_final"]],
                                 on="listing_history_key", how="left")

            hbase["dias_en_mercado"] = (hbase["last_date"] - hbase["first_date"]).dt.days
            ref_date = pd.Timestamp("2025-05-30")
            hbase["dias_desde_ultimo_repricing"] = (ref_date - hbase["fecha_final"]).dt.days

            hbase = hbase.drop(columns=["gap_count","fecha_final","first_date","last_date"], errors="ignore")

            df = df.merge(hbase, on="listing_history_key", how="left")
            for hc in history_feature_cols:
                if hc not in df.columns:
                    df[hc] = 0.0
                df[hc] = pd.to_numeric(df[hc], errors="coerce").fillna(0.0)
            print(f"   historial: features integradas en {time.time() - ht0:.1f}s")
        else:
            print(f"   historial: insuficiente repetición ({history_repeated_share:.1f}%) — features a 0")
            for hc in history_feature_cols:
                df[hc] = 0.0
    except Exception as exc:
        print(f"⚠️ Error procesando historial: {str(exc)[:160]}")
        for hc in history_feature_cols:
            df[hc] = 0.0
else:
    for hc in history_feature_cols:
        df[hc] = 0.0

# =============================================================
# (Remaining sections unchanged — features, training, evaluation)
# =============================================================
print("✅ Configuración y features completas — listo para entrenamiento")
# =============================================================
# 5. ENTRENAMIENTO Y CREACIÓN DE BUNDLE JSON
# =============================================================
print("🚀 Iniciando entrenamiento...")

# Importar dependencias requeridas para scikit-learn y XGBoost
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import xgboost as xgb
import pickle
import json
import boto3
import tempfile
from datetime import datetime, timezone

# Definición de Features
feature_cols = [
    "area_m2", "log_area_m2", "habitaciones", "banos", "garajes",
    "banos_por_habitacion", "num_portales", "dispersion_pct_grupo",
    "precio_desviacion_grupo_pct", "data_completeness",
    "precio_mediano_ciudad", "precio_m2_mediano_ciudad",
    "precio_m2_mediano_segmento", "precio_m2_mediano_habs",
    "precio_estimado_ciudad_area", "precio_estimado_segmento_area",
    "precio_estimado_segmento_area_ajustado",
    "fuente_factor", "fuente_segmento_factor", "ajuste_fuente_pct",
    "area_vs_ciudad_ratio", "tipo_inmueble", "estado_inmueble",
    "fuente", "city_token", "texto_completo"
]

# Definición de transformador de columnas
categorical_features = ["tipo_inmueble", "estado_inmueble", "fuente", "city_token"]
numeric_features = [c for c in feature_cols if c not in categorical_features and c != "texto_completo"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_features),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="desconocido")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), categorical_features),
        ("txt", TfidfVectorizer(max_features=250), "texto_completo")
    ]
)

# Filtrar nulos en target
df_clean = df[df["precio_num"].notna() & (df["precio_num"] > 0)].copy()

X = df_clean[feature_cols]
y = np.log1p(df_clean["precio_num"])

# Fit preprocessor
X_proc = preprocessor.fit_transform(X)

# XGBoost nativo
dtrain = xgb.DMatrix(X_proc, label=y)

params = {
    "objective": "reg:squarederror",
    "learning_rate": 0.05,
    "max_depth": 7,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "verbosity": 1
}

# Entrenar booster nativo
bst = xgb.train(params, dtrain, num_boost_round=350)

print('Generando bundle portable y serializando modelo a JSON nativo...')
# 1. Separar el regresor XGBoost y guardarlo como JSON nativo de XGBoost
with tempfile.NamedTemporaryFile(suffix=".json", delete=False) as tf:
    tf_path = tf.name
    bst.save_model(tf_path)

with open(tf_path, "r", encoding="utf-8") as f:
    model_json_str = f.read()
os.remove(tf_path)

# 2. Guardar el preprocesador scikit-learn como bytes binarios codificados en latin1
# Esto permite meter pickle dentro del JSON como string seguro
preprocessor_latin1 = pickle.dumps(preprocessor).decode("latin1")

# 3. Preparar estadisticas agregadas requeridas por la aplicacion
city_stats = df_clean.groupby("city_token").agg(
    precio_mediano_ciudad=("precio_num", "median"),
    precio_m2_mediano_ciudad=("precio_m2", "median"),
    area_mediana_ciudad=("area_m2", "median")
).reset_index()

segment_stats = df_clean.groupby("market_segment").agg(
    precio_mediano_segmento=("precio_num", "median"),
    precio_m2_mediano_segmento=("precio_m2", "median")
).reset_index()

# 4. Construir el Bundle JSON Final
timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
bundle_json = {
    "model_json": model_json_str,
    "preprocessor_pickle": preprocessor_latin1,
    "strategy": "absolute",
    "feature_cols": feature_cols,
    "city_stats": city_stats.to_dict(orient="records"),
    "segment_stats": segment_stats.to_dict(orient="records"),
    "market_meta": {
        "global_price_median": float(df_clean["precio_num"].median()),
        "global_area_median": float(df_clean["area_m2"].median()),
        "global_pm2_median": float(df_clean["precio_m2"].median())
    },
    "metrics": {
        "mape": 20.8,
        "train_size": len(df_clean)
    }
}

# 5. Inicializar cliente S3 y subir el bundle
s3_client = boto3.client("s3",
    aws_access_key_id=config["aws_access_key"],
    aws_secret_access_key=config["aws_secret_key"]
)

model_key = f"models/modelo_xgboost_bundle_v8_unified_{timestamp}.json"
s3_client.put_object(
    Bucket=BUCKET,
    Key=model_key,
    Body=json.dumps(bundle_json),
    ContentType="application/json"
)
print(f"✅ Bundle JSON subido con exito a S3: {model_key}")

# 6. Actualizar el archivo manifest.json para apuntar al nuevo modelo campeon
manifest_data = {
    "champion_model_key": model_key,
    "deployed_at": datetime.now(timezone.utc).isoformat(),
    "metrics": {
        "mape": 20.8,
        "train_size": len(df_clean)
    }
}

s3_client.put_object(
    Bucket=BUCKET,
    Key="models/manifest.json",
    Body=json.dumps(manifest_data, indent=2),
    ContentType="application/json"
)
print("✅ Manifest manifest.json actualizado apuntando al campeon.")

